In [1]:
import pandas as pd

df = pd.read_csv("hf://datasets/bitext/Bitext-customer-support-llm-chatbot-training-dataset/Bitext_Sample_Customer_Support_Training_Dataset_27K_responses-v11.csv")

/home/farhaan/dev/portfolio_projects/llm-customer-support/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df.sample(n=100, random_state=42)

,flags,instruction,category,intent,response
9329,BLZ,I can't talk with a human agent,CONTACT,contact_human_agent,Thank you for reaching out! I'm sorry to hear ...
4160,BLMZ,I have got to locate hte bills from {{Person N...,INVOICE,check_invoice,I'll get right on it! I understand your need t...
18500,BCELM,"I cannot pay, help me to inform of a problem w...",PAYMENT,payment_issue,I apologize for the inconvenience caused by th...
8840,BL,I want help speaking to customer service,CONTACT,contact_customer_service,Grateful for your contact! I get the sense tha...
5098,BLZ,I try to see th accepted payment options,PAYMENT,check_payment_methods,Thank you for reaching out! I completely under...
...,...,...,...,...,...
8489,BLQ,i dont know how i can talk to customer service,CONTACT,contact_customer_service,Thanks for reaching out! I understand that you...
10061,BL,help me to create a {{Account Category}} account,ACCOUNT,create_account,How wonderful! I'm here to guide you through t...
21734,BL,I try to notify of a sign-up error,ACCOUNT,registration_problems,We appreciate your effort in notifying us of t...
196,BMQ,i have problems with canceling order {{Order N...,ORDER,cancel_order,I've grasped that you're experiencing issues w...


In [3]:
dev_df = df.sample(n=100, random_state=42)

In [4]:
dev_df.drop(columns=['flags', 'category', 'response'],inplace=True)

In [5]:
dev_df

,instruction,intent
9329,I can't talk with a human agent,contact_human_agent
4160,I have got to locate hte bills from {{Person N...,check_invoice
18500,"I cannot pay, help me to inform of a problem w...",payment_issue
8840,I want help speaking to customer service,contact_customer_service
5098,I try to see th accepted payment options,check_payment_methods
...,...,...
8489,i dont know how i can talk to customer service,contact_customer_service
10061,help me to create a {{Account Category}} account,create_account
21734,I try to notify of a sign-up error,registration_problems
196,i have problems with canceling order {{Order N...,cancel_order


In [6]:
pd.unique(df['intent'])

array(['cancel_order', 'change_order', 'change_shipping_address',
       'check_cancellation_fee', 'check_invoice', 'check_payment_methods',
       'check_refund_policy', 'complaint', 'contact_customer_service',
       'contact_human_agent', 'create_account', 'delete_account',
       'delivery_options', 'delivery_period', 'edit_account',
       'get_invoice', 'get_refund', 'newsletter_subscription',
       'payment_issue', 'place_order', 'recover_password',
       'registration_problems', 'review', 'set_up_shipping_address',
       'switch_account', 'track_order', 'track_refund'], dtype=object)

In [7]:
len(pd.unique(dev_df['intent']))

26

In [8]:
from pydantic import BaseModel

In [9]:
from enum import Enum

In [10]:
INTENTS_LIST = ['cancel_order', 'change_order', 'change_shipping_address',
       'check_cancellation_fee', 'check_invoice', 'check_payment_methods',
       'check_refund_policy', 'complaint', 'contact_customer_service',
       'contact_human_agent', 'create_account', 'delete_account',
       'delivery_options', 'delivery_period', 'edit_account',
       'get_invoice', 'get_refund', 'newsletter_subscription',
       'payment_issue', 'place_order', 'recover_password',
       'registration_problems', 'review', 'set_up_shipping_address',
       'switch_account', 'track_order', 'track_refund']

In [11]:
# class Intent(Enum):
#     CANCEL_ORDER: 'cancel_order'
#     CHANGE_ORDER: 'change_order'
#     CHANGE_SHIPPING_ADDRESS: 'change_shipping_address'
#     CHECK_CANCELLATION_FEE: 'check_cancellation_fee'
#     CHECK_INVOICE: 'check_invoice'
#     CHECK_PAYMENT_METHODS: 'check_payment_methods'
#     CHECK_REFUND_POLICY: 'check_refund_policy'
#     COMPLAINT: 'complaint'
#     CONTACT_CUSTOMER_SERVICE: 'contact_customer_service'
#     CONTACT_HUMAN_AGENT: 'contact_human_agent'
#     CREATE_ACCOUNT: 'create_account'
#     DELETE_ACCOUNT: 'delete_account'
#     DELIVERY_OPTIONS: 'delivery_options'
#     DELIVERY_PERIOD: 'delivery_period'
#     EDIT_ACCOUNT: 'edit_account'
#     GET_INVOICE: 'get_invoice'
#     GET_REFUND: 'get_refund'
#     NEWSLETTER_SUBSCRIPTION: 'newsletter_subscription'
#     PAYMENT_ISSUE: 'payment_issue'
#     PLACE_ORDER: 'place_order'
#     RECOVER_PASSWORD: 'recover_password'
#     REGISTRATION_PROBLEMS: 'registration_problems'
#     REVIEW: 'review'
#     SET_UP_SHIPPING_ADDRESS: 'set_up_shipping_address'
#     SWITCH_ACCOUNT: 'switch_account'
#     TRACK_ORDER: 'track_order'
#     TRACK_REFUND: 'track_refund'

In [12]:
for item in dev_df.iterrows():
    test = item
    break

In [13]:
ins = test[-1].instruction

In [15]:
from typing import Literal
from pydantic import BaseModel, ValidationError

from ollama import chat


# Define the schema for the response
class Ticket(BaseModel):
    description: str
    # TODO: Find a cleaner way to do this
    intent: Literal['cancel_order', 'change_order', 'change_shipping_address',
       'check_cancellation_fee', 'check_invoice', 'check_payment_methods',
       'check_refund_policy', 'complaint', 'contact_customer_service',
       'contact_human_agent', 'create_account', 'delete_account',
       'delivery_options', 'delivery_period', 'edit_account',
       'get_invoice', 'get_refund', 'newsletter_subscription',
       'payment_issue', 'place_order', 'recover_password',
       'registration_problems', 'review', 'set_up_shipping_address',
       'switch_account', 'track_order', 'track_refund']

response = chat(
  model='qwen3.5:2b',
  messages=[{'role': 'user', 'content': f'Classify the ticket into intent based on the description. Here is the list of allowed intents: {str(INTENTS_LIST)}. Here is the description to classify: {ins}. Return the description and most likely intent.'}],
  format=Ticket.model_json_schema(),  # Use Pydantic to generate the schema or format=schema
  options={'temperature':0},  # Make responses more deterministic
    think=False
)


try:
    # Use Pydantic to validate the response
    res = Ticket.model_validate_json('{"test":"none"}')
except ValidationError as exc:
    # res = Ticket.model_validate_json(response.message.content)
    res = {'description': ins, "intent": 'manual_review'}
print(res)

{'description': "I can't talk with  a human agent", 'intent': 'manual_review'}


In [28]:
dev_df.columns

Index(['instruction', 'intent'], dtype='object')

In [44]:
class Person(BaseModel):
    name: str
    age: int


response = chat(
    model="qwen3.5:2b",
    messages=[
        {
            "role": "user",
            "content": "John is 32 years old. Return his name and age."
        }
    ],
    format=Person.model_json_schema(),
    think=False,
)

print(response.message.content)

person = Person.model_validate_json(response.message.content)

{
"name": "John",
"age": 32
}
